In [19]:
import os, glob, time, unicodedata
import numpy as np, pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/fever_project'
except ImportError:
    ROOT = './fever_project'          # running outside Colab
if not os.path.isdir(ROOT):
    raise FileNotFoundError(f'{ROOT} not found. Add the shared fever_project folder as a shortcut in My Drive.')
for d in ['data/evidence', 'data/processed', 'results']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

LABELS = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']   # fixed label order
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
CONTRACT_COLS = ['id', 'claim', 'evidence', 'label', 'label_id', 'evidence_source', 'n_sent']

def nfc(s):
    return unicodedata.normalize('NFC', s)

MINI = False                     # True = test on a sample of NEI claims, False = full build
MINI_N = 2000                    # NEI claims per split in MINI mode
SEED = 42
CHUNK = 200                      # lower to 50 if Colab runs out of RAM
EXCLUDE_SHARED_FROM_TRAIN = True # pages needed by BOTH splits are kept out of the train pool

# Length matching (fixes NEI passages being shorter than gold evidence)
LENGTH_MATCH = True              # False = plain TF-IDF top-k (the old behaviour)
LEN_ALPHA = 0.7                  # strength of the length preference: 0 = none, higher = stricter
CAND = 50                        # candidate sentences (most similar) considered per claim
MIN_WORDS = 6                    # drop tiny fragments from the pools
print('ROOT:', ROOT, '| MINI:', MINI, '| LENGTH_MATCH:', LENGTH_MATCH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/fever_project | MINI: False | LENGTH_MATCH: True


In [20]:
cols = ['id', 'label', 'claim', 'evidence_wiki_url']
train_df = pd.read_parquet(f'{ROOT}/data/processed/claims_train_flat.parquet', columns=cols)
val_df = pd.read_parquet(f'{ROOT}/data/processed/claims_val_flat.parquet', columns=cols)
print('rows  train:', len(train_df), '| val:', len(val_df))
print('unique claims  train:', train_df.id.nunique(), '| val:', val_df.id.nunique())
n_nei_tr = train_df[train_df.label == 'NOT ENOUGH INFO'].id.nunique()
n_nei_va = val_df[val_df.label == 'NOT ENOUGH INFO'].id.nunique()
print('NEI claims  train:', n_nei_tr, '| val:', n_nei_va, '(Member 1 reported 35,639 and 3,333)')
if (n_nei_tr, n_nei_va) != (35639, 3333):
    print('WARNING: NEI counts differ from Member 1 report. Check that you loaded the *_flat files.')

rows  train: 311431 | val: 18999
unique claims  train: 145449 | val: 9999
NEI claims  train: 35639 | val: 3333 (Member 1 reported 35,639 and 3,333)


In [21]:
def needed_pages(df):
    d = df[(df.label != 'NOT ENOUGH INFO') & (df.evidence_wiki_url != '')]    # SUPPORTS/REFUTES rows that point to a page
    return set(d.evidence_wiki_url.map(nfc))

need_train, need_val = needed_pages(train_df), needed_pages(val_df)
shared = need_train & need_val
print('pages needed: train', len(need_train), '| val', len(need_val), '| shared', len(shared))

parts = sorted(glob.glob(f'{ROOT}/data/evidence/part_*.parquet'))
assert parts, 'No part_*.parquet in data/evidence. Ask Member 1.'
print('part files found:', len(parts), '(Member 1 made 11: part_0 ... part_10)')
sent = pd.concat([pd.read_parquet(f) for f in parts], ignore_index=True)
sent = sent.drop_duplicates(['page', 'sent_id'])
print('sentences loaded:', len(sent), '| pages:', sent.page.nunique())
still_missing = (need_train | need_val) - set(sent.page)
print('required pages missing from sentence files:', len(still_missing))
assert not still_missing, 'Some required pages are missing. Make sure part_10.parquet exists.'

# Pools. Train NEI may only search train pages, validation NEI only validation pages.
# 451 pages are needed by BOTH splits. With EXCLUDE_SHARED_FROM_TRAIN they appear only in the
# validation pool, so no validation page is ever used as training NEI evidence.
train_pages = (need_train - shared) if EXCLUDE_SHARED_FROM_TRAIN else need_train
pool_train = sent[sent.page.isin(train_pages)]
pool_val = sent[sent.page.isin(need_val)]
print('pool sizes (sentences)  train:', len(pool_train), '| val:', len(pool_val))
assert len(pool_train) > 0 and len(pool_val) > 0, 'Empty pool'

def load_gold(name):
    # returns (number of sentences, number of words) of every gold evidence passage
    p = f'{ROOT}/data/processed/gold_{name}.parquet'
    if os.path.exists(p):
        g = pd.read_parquet(p, columns=['n_sent', 'evidence'])
        return g.n_sent.to_numpy(), g.evidence.str.split().str.len().to_numpy()
    if MINI:
        print(f'WARNING: {p} missing, using a placeholder (MINI only)')
        return np.array([1, 1, 1, 2, 2, 3]), np.array([25, 30, 35, 60, 70, 90])
    raise FileNotFoundError(p)

gold_info = {'train': load_gold('train'), 'val': load_gold('val')}     # real evidence shapes for both splits
for n, (k_, w_) in gold_info.items():
    print(f'gold {n}: mean sentences {k_.mean():.2f} | mean words {w_.mean():.1f} | median words {np.median(w_):.0f}')

pages needed: train 12549 | val 1460 | shared 451
part files found: 11 (Member 1 made 11: part_0 ... part_10)
sentences loaded: 130699 | pages: 15315
required pages missing from sentence files: 0
pool sizes (sentences)  train: 104163 | val: 13241
gold train: mean sentences 1.86 | mean words 57.4 | median words 36
gold val: mean sentences 1.67 | mean words 47.4 | median words 32


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_nei(df, pool, gold_k, gold_w, seed=42, chunk=200, max_claims=None,
              length_match=True, alpha=0.7, cand=50, min_words=6):
    nei = (df[df.label == 'NOT ENOUGH INFO'].drop_duplicates('id')[['id', 'claim', 'label']]
           .reset_index(drop=True))
    if max_claims is not None and len(nei) > max_claims:
        nei = nei.sample(max_claims, random_state=seed).reset_index(drop=True)
    pool = pool.drop_duplicates(['page', 'sent_id']).reset_index(drop=True)
    wl = pool['text'].str.split().str.len().to_numpy()
    keep = wl >= min_words                    # drop tiny fragments
    pool, wl = pool[keep].reset_index(drop=True), wl[keep]
    print(f'  pool after dropping sentences under {min_words} words: {len(pool)}')
    pool_text = pool['text'].to_numpy()

    vec = TfidfVectorizer(stop_words='english', sublinear_tf=True, min_df=2, max_df=0.5, dtype=np.float32)
    Xs = vec.fit_transform(pool_text)         # sentence weights
    Xc = vec.transform(nei['claim'])          # claim weights, same vocabulary
    XsT = Xs.T.tocsr()

    # For each NEI claim copy the SHAPE of one real gold example: how many sentences (k) and how many
    # words in total. Sampling a whole gold row keeps sentence count and word count linked.
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(gold_k), size=len(nei))
    ks = np.maximum(gold_k[idx].astype(int), 1)
    target = np.maximum(gold_w[idx] / ks, 5.0)    # wanted words per sentence

    evid, cnt = [], []
    t0 = time.time()
    for s in range(0, len(nei), chunk):       # chunks keep memory small
        sim = (Xc[s:s + chunk] @ XsT).tocsr()
        for r in range(sim.shape[0]):
            a, b = sim.indptr[r], sim.indptr[r + 1]
            data, ind = sim.data[a:b], sim.indices[a:b]
            k = int(ks[s + r])
            if length_match and len(data) > 0:
                C = max(cand, 5 * k)
                if len(data) > C:
                    csel = np.argpartition(-data, C - 1)[:C]      # the C most similar sentences
                else:
                    csel = np.arange(len(data))
                cdata, cind = data[csel], ind[csel]
                # similarity x length preference: 1.0 when the sentence length equals the target,
                # smaller the further away it is (too short OR too long)
                pen = np.exp(-alpha * np.abs(np.log(np.maximum(wl[cind], 1) / target[s + r])))
                order = np.argsort(-(cdata * pen), kind='stable')[:k]
                top = cind[order]
            else:
                if len(data) > k:
                    sel = np.argpartition(-data, k - 1)[:k]
                else:
                    sel = np.arange(len(data))
                sel = sel[np.argsort(-data[sel], kind='stable')]
                top = ind[sel]
            evid.append(' '.join(pool_text[top]))
            cnt.append(len(top))
        if (s // chunk) % 25 == 0:
            print(f'  {min(s + chunk, len(nei))}/{len(nei)} claims | {time.time() - t0:.0f}s')
    nei['evidence'], nei['n_sent'], nei['evidence_source'] = evid, cnt, 'tfidf'
    nei['label_id'] = label2id['NOT ENOUGH INFO']
    before = len(nei)
    nei = nei[nei.evidence.str.len() > 0]
    print(f'kept {len(nei)} of {before} NEI claims (dropped {before - len(nei)} with no similar sentence)')
    return nei[CONTRACT_COLS].reset_index(drop=True)

In [23]:
mc = MINI_N if MINI else None
opts = dict(seed=SEED, chunk=CHUNK, max_claims=mc, length_match=LENGTH_MATCH,
            alpha=LEN_ALPHA, cand=CAND, min_words=MIN_WORDS)
print('--- train ---')
nei_train = build_nei(train_df, pool_train, *gold_info['train'], **opts)
print('--- validation ---')
nei_val = build_nei(val_df, pool_val, *gold_info['val'], **opts)
print(len(nei_train), len(nei_val))
print(nei_train.sample(3, random_state=0)[['claim', 'evidence', 'n_sent']].to_string(max_colwidth=120))

--- train ---
  pool after dropping sentences under 6 words: 103538
  200/35639 claims | 0s
  5200/35639 claims | 1s
  10200/35639 claims | 1s
  15200/35639 claims | 2s
  20200/35639 claims | 2s
  25200/35639 claims | 3s
  30200/35639 claims | 4s
  35200/35639 claims | 4s
kept 35638 of 35639 NEI claims (dropped 1 with no similar sentence)
--- validation ---
  pool after dropping sentences under 6 words: 13149
  200/3333 claims | 0s
kept 3331 of 3333 NEI claims (dropped 2 with no similar sentence)
35638 3331
                                                                      claim                                                                                                                 evidence  n_sent
31852                           Planet of the Apes shot scenes in Colorado.                It was adapted into the 1968 film Planet of the Apes , launching the Planet of the Apes media franchise .       1
276                   Tyrion Lannister is also referred to as "the potato."  

In [24]:
def words(s):
    return s.str.split().str.len()

for name, d in [('train', nei_train), ('val', nei_val)]:
    gk, gw = gold_info[name]
    nw = words(d.evidence)
    ratio = nw.mean() / gw.mean()
    print(f'[{name}] NEI claims: {len(d)} | unique ids: {d.id.is_unique} | empty evidence: {int((d.evidence.str.len() == 0).sum())}')
    print(f'   mean n_sent  NEI: {d.n_sent.mean():.2f} | gold: {gk.mean():.2f}')
    print(f'   mean words   NEI: {nw.mean():.1f} | gold: {gw.mean():.1f} | ratio NEI/gold: {ratio:.2f}')
    print(f'   words 25/50/75%  NEI: {np.percentile(nw, [25, 50, 75]).round(0)} | gold: {np.percentile(gw, [25, 50, 75]).round(0)}')
    if not 0.85 <= ratio <= 1.15:
        print('WARNING: NEI and gold lengths still differ by more than 15%. Raise LEN_ALPHA or CAND, or tell the group.')

bad = sum(int(d.evidence.str.contains('-LRB-|-RRB-|-LSB-|-RSB-|-COLON-').sum()) for d in (nei_train, nei_val))
print('NEI evidence with leftover FEVER bracket tokens:', bad)

# NEI claim text that is also the text of a SUPPORTS/REFUTES claim in the same split
# (conflicting labels for identical text; report this number to the group)
for name, raw, d in [('train', train_df, nei_train), ('val', val_df, nei_val)]:
    sr_text = set(raw[raw.label != 'NOT ENOUGH INFO'].claim)
    print(f'[{name}] NEI claims whose exact text also appears as a SUPPORTS/REFUTES claim:', int(d.claim.isin(sr_text).sum()))

# sanity: validation NEI evidence must come only from validation pages
assert set(pool_val.page) <= need_val and set(pool_train.page) <= need_train

[train] NEI claims: 35638 | unique ids: True | empty evidence: 0
   mean n_sent  NEI: 1.87 | gold: 1.86
   mean words   NEI: 50.2 | gold: 57.4 | ratio NEI/gold: 0.87
   words 25/50/75%  NEI: [22. 31. 50.] | gold: [25. 36. 59.]
[val] NEI claims: 3331 | unique ids: True | empty evidence: 0
   mean n_sent  NEI: 1.63 | gold: 1.67
   mean words   NEI: 41.7 | gold: 47.4 | ratio NEI/gold: 0.88
   words 25/50/75%  NEI: [20. 28. 43.] | gold: [22. 32. 49.]
NEI evidence with leftover FEVER bracket tokens: 0
[train] NEI claims whose exact text also appears as a SUPPORTS/REFUTES claim: 1101
[val] NEI claims whose exact text also appears as a SUPPORTS/REFUTES claim: 35


In [25]:
suffix = '_mini' if MINI else ''
nei_train.to_parquet(f'{ROOT}/data/processed/nei_train{suffix}.parquet', index=False)
nei_val.to_parquet(f'{ROOT}/data/processed/nei_val{suffix}.parquet', index=False)
print('saved nei_train%s.parquet and nei_val%s.parquet' % (suffix, suffix))

audit_path = f'{ROOT}/results/nei_audit_train.csv'
if not MINI:
    if os.path.exists(audit_path):
        print('Audit sheet already exists, NOT overwritten (someone may have filled it):', audit_path)
    else:
        audit = nei_train.sample(50, random_state=1)[['id', 'claim', 'evidence']].copy()
        for c in ['judgement_m1', 'judgement_m2', 'judgement_m3', 'notes']:
            audit[c] = ''     # fill by hand with: neither / supports / refutes
        audit.to_csv(audit_path, index=False)
        print('audit sheet written:', audit_path)

saved nei_train.parquet and nei_val.parquet
audit sheet written: /content/drive/MyDrive/fever_project/results/nei_audit_train.csv
